# Compare models: Decision-1 and Jev

The same accounts go to both models with **identical input** (exactly what Phase 2 sends) and the same questions.
Each model's scores then go through the same rules (`decide.py`), with its own thresholds if set below, so the
model is the only difference. Nothing is written to the crawl store.

Keys come from the `.env` file (`DECISION1_API_KEY`, `JEV_API_KEY`). Safe to stop and re-run: answered calls are
skipped and failed ones retried.

## Settings

In [ ]:
from pathlib import Path
import compare, crawl

INPUT   = Path("data/compare_20each.xlsx")           # 24 fresh accounts per segment label (ZoomInfo)
STORE   = Path("data/crawls_compare_20each.jsonl")   # this set's own crawl store
ANSWERS = Path("data/compare_answers_20each.csv")    # one line per call, both models
REPORT  = Path("data/compare_report_20each.xlsx")    # also writes compare_report_..._accounts.csv
LABEL   = "ZoomInfo"   # what the input's industry_types column is; "" = nothing to compare with
MODELS  = ("decision1", "jev")
# A model's own thresholds. Jev is less sure about true Others than Decision-1, so its Other bar is lower: Other only
# when p_other is 0.70 or more AND every main-business score is under 0.40. Decision-1 uses decide.py's (0.80, 0.50).
THRESHOLDS = {"jev": {"other_from": 0.70, "main_blocks_other": 0.40}}
WORKERS = 4
LIMIT   = None         # at most this many accounts (e.g. 3 to try it out)

## 1. Crawl the set (Phase 1)

In [ ]:
crawl.run(INPUT, STORE)

## 2. Ask both models

In [ ]:
compare.run(STORE, INPUT, ANSWERS, models=MODELS, workers=WORKERS, limit=LIMIT)

## 3. Report
| sheet | what's on it |
|---|---|
| Guide | every column, the colours, each model's thresholds |
| Summary | how often the models agree; each model's industries, how sure, match with the label; for each model, how many accounts are Other at each pair of thresholds (to tune them) |
| Disagreements | the accounts where the models pick different industries, with both rationales: the ones to look at |
| All accounts | every account side by side: industries, then confidences, then main business, then rationales |

Changing `THRESHOLDS` needs only this cell again, not new calls.

In [ ]:
compare.report(ANSWERS, REPORT, LABEL, MODELS, THRESHOLDS)
table = compare.side_by_side(ANSWERS, LABEL, MODELS, THRESHOLDS)
table[table["Models agree"] == "No"][["Account", f"{LABEL} industry", "Decision-1", "Jev"]]